# GraphCast — Baseline Results
Visualizes RMSE and ACC from `results/baselines.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("results")

df = pd.read_csv(RESULTS_DIR / "baselines.csv")

# Parse lead_time to hours for sorting
def lead_to_hours(s):
    s = str(s)
    if 'd' in s and 'h' in s:
        d, h = s.split('d')
        return int(d) * 24 + int(h.replace('h', ''))
    elif 'd' in s:
        return int(s.replace('d', '')) * 24
    else:
        return int(s.replace('h', ''))

df['lead_hours'] = df['lead_time'].apply(lead_to_hours)
df = df.sort_values(['model', 'variable', 'level', 'lead_hours'])

models   = df['model'].unique()
df_surf  = df[df['level'] == 'all']
vars_all = sorted(df_surf['variable'].unique())

COLORS = {
    'graphcast_pretrained': '#1f77b4',
    'persistence':          '#d62728',
    'finetuned':            '#2ca02c',
}
LINESTYLES = {
    'graphcast_pretrained': '-',
    'persistence':          '--',
    'finetuned':            '-.',
}

def color(m):     return COLORS.get(m, '#7f7f7f')
def linestyle(m): return LINESTYLES.get(m, ':')

print(f"Models   : {list(models)}")
print(f"Variables: {vars_all}")
print(f"Lead times: {sorted(df_surf['lead_hours'].unique())} hours")
df.head()

## RMSE vs Lead Time — surface variables

In [ ]:
ncols = 3
nrows = int(np.ceil(len(vars_all) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, var in zip(axes, vars_all):
    for model in models:
        sub = df_surf[(df_surf['variable'] == var) & (df_surf['model'] == model)]
        if sub.empty:
            continue
        ax.plot(
            sub['lead_hours'], sub['rmse'],
            label=model, color=color(model), ls=linestyle(model), marker='o', ms=4,
        )
    ax.set_title(var, fontsize=10)
    ax.set_xlabel('Lead time (h)')
    ax.set_ylabel('RMSE')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for ax in axes[len(vars_all):]:
    ax.set_visible(False)

fig.suptitle('RMSE vs Lead Time (latitude-weighted, Eq. 20)', fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'rmse_vs_leadtime.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/rmse_vs_leadtime.png')

## RMSE vs Pressure Level — 3-D variables

In [ ]:
df_3d = df[df['level'] != 'all'].copy()
df_3d['level'] = df_3d['level'].astype(int)
vars_3d = sorted(df_3d['variable'].unique())

if not vars_3d:
    print("No 3-D variables found.")
else:
    # One subplot per 3-D variable; lines = models, one representative lead time
    lead_times_avail = sorted(df_3d['lead_hours'].unique())
    # Pick up to 3 lead times spread across the range
    idxs = np.round(np.linspace(0, len(lead_times_avail) - 1, min(3, len(lead_times_avail)))).astype(int)
    selected_leads = [lead_times_avail[i] for i in idxs]

    ncols = min(3, len(vars_3d))
    nrows = int(np.ceil(len(vars_3d) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    for ax, var in zip(axes, vars_3d):
        for model in models:
            for i, lh in enumerate(selected_leads):
                sub = df_3d[
                    (df_3d['variable'] == var) &
                    (df_3d['model'] == model) &
                    (df_3d['lead_hours'] == lh)
                ].sort_values('level')
                if sub.empty:
                    continue
                label = f"{model} {lh}h" if i == 0 else f"_ {lh}h"
                ax.plot(
                    sub['rmse'], sub['level'],
                    label=label, color=color(model),
                    ls=['-', '--', ':'][i], marker='o', ms=3,
                )
        ax.invert_yaxis()
        ax.set_title(var, fontsize=10)
        ax.set_xlabel('RMSE')
        ax.set_ylabel('Pressure level (hPa)')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    for ax in axes[len(vars_3d):]:
        ax.set_visible(False)

    fig.suptitle('RMSE vs Pressure Level', fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'rmse_vs_level.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → results/rmse_vs_level.png')

## Skill Score RMSE — pretrained vs persistence

In [ ]:
skill_path = RESULTS_DIR / 'skill_scores_pretrained_vs_persistence.csv'

if not skill_path.exists():
    print(f"{skill_path} not found — run run_metrics.py first.")
else:
    sk = pd.read_csv(skill_path)
    sk['lead_hours'] = sk['lead_time'].apply(lead_to_hours)
    sk = sk[sk['level'] == 'all'].sort_values(['variable', 'lead_hours'])
    vars_sk = sorted(sk['variable'].unique())

    ncols = 3
    nrows = int(np.ceil(len(vars_sk) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for ax, var in zip(axes, vars_sk):
        sub = sk[sk['variable'] == var]
        ax.axhline(0, color='gray', lw=0.8, ls='--')
        ax.plot(sub['lead_hours'], sub['skill_rmse'],
                color='#1f77b4', marker='o', ms=4, label='skill RMSE')
        ax.set_title(var, fontsize=10)
        ax.set_xlabel('Lead time (h)')
        ax.set_ylabel('Skill RMSE\n(neg = better than persistence)')
        ax.grid(True, alpha=0.3)

    for ax in axes[len(vars_sk):]:
        ax.set_visible(False)

    fig.suptitle('Normalized RMSE Skill Score\n(graphcast_pretrained vs persistence)', fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'skill_rmse.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → results/skill_rmse.png')

## Summary Table

In [ ]:
summary = (
    df_surf
    .groupby(['model', 'variable'])[['rmse']]
    .mean()
    .round(4)
    .unstack('model')
)
summary.columns = summary.columns.droplevel(0)
summary

## Prediction Maps (spot check)

In [ ]:
import xarray

preds_path   = RESULTS_DIR / 'preds_graphcast_pretrained.nc'
targets_path = RESULTS_DIR / 'eval_targets.nc'

if not preds_path.exists() or not targets_path.exists():
    print("Prediction/target NetCDF files not found — run run_baseline.py first.")
else:
    preds   = xarray.load_dataset(preds_path)
    targets = xarray.load_dataset(targets_path)

    # Pick a surface variable to plot
    SURF_VARS = ['2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind']
    plot_var = next((v for v in SURF_VARS if v in preds.data_vars), list(preds.data_vars)[0])
    t_idx = 0  # first lead time

    pred_slice = preds[plot_var].isel(time=t_idx)
    tgt_slice  = targets[plot_var].isel(time=t_idx)
    if 'batch' in pred_slice.dims: pred_slice = pred_slice.isel(batch=0)
    if 'batch' in tgt_slice.dims:  tgt_slice  = tgt_slice.isel(batch=0)

    vmin = float(min(pred_slice.min(), tgt_slice.min()))
    vmax = float(max(pred_slice.max(), tgt_slice.max()))

    lead_str = str(preds.coords['time'].values[t_idx])

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].imshow(tgt_slice.values,  origin='upper', cmap='RdBu_r', vmin=vmin, vmax=vmax)
    axes[0].set_title(f'Target — {plot_var}')

    im = axes[1].imshow(pred_slice.values, origin='upper', cmap='RdBu_r', vmin=vmin, vmax=vmax)
    axes[1].set_title(f'GraphCast pretrained (lead {lead_str})')
    fig.colorbar(im, ax=axes[1])

    err = pred_slice.values - tgt_slice.values
    elim = float(np.abs(err).max())
    im2 = axes[2].imshow(err, origin='upper', cmap='bwr', vmin=-elim, vmax=elim)
    axes[2].set_title('Error (pred − target)')
    fig.colorbar(im2, ax=axes[2])

    for ax in axes:
        ax.axis('off')

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f'map_{plot_var}_t0.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → results/map_{plot_var}_t0.png')